# Mina Dog Vocalization Classifier — Training

2-class model: **mina** (dog vocalizations) vs **negative** (ambient noise, speech, etc.)

Exports to TFLite for Raspberry Pi 5 inference (<25ms).

## Setup
1. Run cells in order
2. Training data downloads automatically from Google Drive
3. Download `mina_classifier.tflite` when done

In [ ]:
# Install dependencies
!pip install -q librosa soundfile scikit-learn gdown

In [ ]:
# Download training data from Google Drive
import zipfile, os

if not os.path.exists('training_data'):
    !gdown --id 1l_KQ7NJppvTb4g_QUSOZTi6SwsocR27n -O training_data.zip
    with zipfile.ZipFile('training_data.zip', 'r') as z:
        z.extractall('.')
    os.remove('training_data.zip')  # Free up disk space
    print('Extracted and cleaned up zip.')
else:
    print('training_data already exists, skipping download.')

for d in ['training_data/mina', 'training_data/negative']:
    count = len([f for f in os.listdir(d) if f.endswith('.wav')]) if os.path.isdir(d) else 0
    print(f'{d}: {count} files')

In [ ]:
import os
import gc
import random
from pathlib import Path

import numpy as np
import librosa
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Config
SAMPLE_RATE = 16000
WINDOW_SIZE = 16000
FRAME_LENGTH = 255
FRAME_STEP = 128
N_MELS = 40
N_MFCC = 40
LABELS = ['mina', 'negative']

print(f'TensorFlow: {tf.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')

In [ ]:
def compute_mfcc(audio):
    """Compute MFCC features. Output shape: (124, 40)"""
    audio = audio.astype(np.float32)
    if len(audio) < WINDOW_SIZE:
        audio = np.pad(audio, (0, WINDOW_SIZE - len(audio)))
    elif len(audio) > WINDOW_SIZE:
        audio = audio[:WINDOW_SIZE]
    mfcc = librosa.feature.mfcc(
        y=audio, sr=SAMPLE_RATE, n_mfcc=N_MFCC,
        n_fft=FRAME_LENGTH, hop_length=FRAME_STEP,
        n_mels=N_MELS, center=False,
    )
    return mfcc.T.astype(np.float32)  # (124, 40)

def load_audio(filepath):
    audio, sr = librosa.load(filepath, sr=SAMPLE_RATE, mono=True)
    if len(audio) < WINDOW_SIZE:
        audio = np.pad(audio, (0, WINDOW_SIZE - len(audio)))
    else:
        audio = audio[:WINDOW_SIZE]
    return audio.astype(np.float32)

# Verify MFCC shape
test = compute_mfcc(np.random.randn(16000).astype(np.float32))
print(f'MFCC shape: {test.shape}')
assert test.shape == (124, 40)

In [ ]:
# Load audio and convert directly to MFCCs (memory efficient)
# Skip keeping raw audio in memory — go straight to features
TRAINING_DATA_DIR = Path('./training_data')

all_mfccs = []
all_labels = []

for label_idx, label_name in enumerate(LABELS):
    label_dir = TRAINING_DATA_DIR / label_name
    if not label_dir.exists():
        print(f'WARNING: {label_dir} not found!')
        continue
    wav_files = sorted(label_dir.glob('*.wav'))
    print(f'Loading {label_name}: {len(wav_files)} files...')
    for i, wav_path in enumerate(wav_files):
        try:
            audio = load_audio(str(wav_path))
            mfcc = compute_mfcc(audio)
            all_mfccs.append(mfcc)
            all_labels.append(label_idx)
        except Exception:
            pass
        if (i + 1) % 2000 == 0:
            print(f'  {i+1}/{len(wav_files)}...')

X_all = np.array(all_mfccs, dtype=np.float32)[..., np.newaxis]  # (N, 124, 40, 1)
y_all = np.array(all_labels, dtype=np.int32)
del all_mfccs, all_labels
gc.collect()

print(f'\nTotal samples: {len(y_all)}')
for i, name in enumerate(LABELS):
    print(f'  {name}: {np.sum(y_all == i)}')
print(f'Shape: {X_all.shape}')

In [ ]:
# Train/val/test split: 70/15/15
X_train, X_temp, y_train, y_temp = train_test_split(
    X_all, y_all, test_size=0.30, random_state=SEED, stratify=y_all
)
del X_all, y_all
gc.collect()

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)
del X_temp, y_temp
gc.collect()

print(f'Train: {len(y_train)}, Val: {len(y_val)}, Test: {len(y_test)}')

In [ ]:
# Data augmentation via TF dataset (no extra memory)
# Apply augmentation on the MFCC features directly

def augment_mfcc(mfcc, label):
    """Apply random augmentation to MFCC features."""
    # Random time shift (roll along time axis)
    shift = tf.random.uniform([], -12, 12, dtype=tf.int32)  # ~10% of 124
    mfcc = tf.roll(mfcc, shift, axis=0)
    # Random noise
    noise = tf.random.normal(tf.shape(mfcc), stddev=0.1)
    mfcc = mfcc + noise
    return mfcc, label

BATCH_SIZE = 32

# Training dataset with augmentation
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = train_ds.shuffle(10000, seed=SEED)
train_ds = train_ds.map(augment_mfcc, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Validation dataset (no augmentation)
val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val))
val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print(f'Train batches: {len(train_ds)}, Val batches: {len(val_ds)}')

In [ ]:
# Class weights
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))
print(f'Class weights: {class_weight_dict}')

# Build model
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(124, 40, 1)),
    tf.keras.layers.Conv2D(8, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(16, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Dense(len(LABELS), activation='softmax'),
])

model.summary()

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

In [ ]:
# Train
callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
# Evaluate
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test accuracy: {test_acc:.4f}')
print(f'Test loss: {test_loss:.4f}')

y_pred = np.argmax(model.predict(X_test), axis=1)

print('\n--- Confusion Matrix ---')
cm = confusion_matrix(y_test, y_pred)
header = '          ' + '  '.join(f'{l:>8s}' for l in LABELS)
print(header)
for i, row in enumerate(cm):
    print(f'{LABELS[i]:>8s}  ' + '  '.join(f'{v:>8d}' for v in row))

print('\n--- Per-class Metrics ---')
print(classification_report(y_test, y_pred, target_names=LABELS, digits=4))

In [ ]:
# Export to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open('mina_classifier.tflite', 'wb') as f:
    f.write(tflite_model)
print(f'TFLite model saved: {len(tflite_model) / 1024:.1f} KB')

with open('labels.txt', 'w') as f:
    for label in LABELS:
        f.write(label + '\n')
print('Labels saved.')

# Verify TFLite model
interpreter = tf.lite.Interpreter(model_path='mina_classifier.tflite')
interpreter.allocate_tensors()
inp = interpreter.get_input_details()
out = interpreter.get_output_details()
print(f'Input: {inp[0]["shape"]}')
print(f'Output: {out[0]["shape"]}')

# Inference benchmark
import time
dummy = np.random.randn(1, 124, 40, 1).astype(np.float32)
times = []
for _ in range(1000):
    t0 = time.perf_counter()
    interpreter.set_tensor(inp[0]['index'], dummy)
    interpreter.invoke()
    times.append((time.perf_counter() - t0) * 1000)
print(f'\nInference benchmark (1000 runs):')
print(f'  Mean: {np.mean(times):.2f} ms')
print(f'  P95:  {np.percentile(times, 95):.2f} ms')

In [ ]:
# Download model files to your machine
from google.colab import files
files.download('mina_classifier.tflite')
files.download('labels.txt')